Scaled dot-product attention from scratch

The core attention mechanism with optional masking

In [6]:
import torch
import torch.nn.functional as F
import math

# scaled dot product attention

def scaled_dot_product_attention(Q, K, V, mask=None):
  """
    Computes attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V

    Args:
        Q: Queries  (batch, seq_len, d_k)
        K: Keys     (batch, seq_len, d_k)
        V: Values   (batch, seq_len, d_v)
        mask: Optional boolean mask (True = attend, False = block)

    Returns:
        output:  (batch, seq_len, d_v)
        weights: (batch, seq_len, seq_len)
  """
  d_k = Q.size(-1)

  # step 1: compute raw attention scores
  scores = torch.matmul(Q, K.transpose(-2,-1)) #  (batch,n, n)

  # step 2: scale by sqrt(d_k) to prevent gradient issues
  scores = scores  / math.sqrt(d_k)

  # step 3: Apply mask (eg causal mask for decoders)
  if mask is not None:
    scores = scores.masked_fill(~mask, float('-inf'))

  # step 4: softmax -> each row sums to 1
  weights = F.softmax(scores, dim=-1)

  # step 5: weighted sum of values
  output = torch.matmul(weights, V)

  return output, weights

batch_size, seq_len, d_model = 1, 6, 64

# simulate token embeddings
X = torch.randn(batch_size, seq_len, d_model)

# Learned projection matrices
W_q = torch.randn(d_model, d_model)
W_k = torch.randn(d_model, d_model)
W_v = torch.randn(d_model, d_model)

# Project input into Q, K, V
Q = X @ W_q  # (1, 6, 64)
K = X @ W_k  # (1, 6, 64)
V = X @ W_v  # (1, 6, 64)

# compute attention bidirectional - no mask
output, weights = scaled_dot_product_attention(Q, K, V)

print(f"Input shape:      {X.shape}")       # [1, 6, 64]
print(f"Output shape:     {output.shape}")   # [1, 6, 64]
print(f"Attention matrix: {weights.shape}")  # [1, 6, 6]
print(f"Row sums to 1:    {weights[0, 0].sum():.4f}")

# causal mask for gpt style
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
# [[True, False, False, ...],
#  [True, True,  False, ...],
#  [True, True,  True,  ...]]

output_causal, weights_causal = scaled_dot_product_attention(
    Q, K, V, mask=causal_mask
)

print(f"\nCausal weights (row 0): {weights_causal[0, 0]}")
# Only first position has non-zero weight


Input shape:      torch.Size([1, 6, 64])
Output shape:     torch.Size([1, 6, 64])
Attention matrix: torch.Size([1, 6, 6])
Row sums to 1:    1.0000

Causal weights (row 0): tensor([1., 0., 0., 0., 0., 0.])
